# Malware Model Training (Offline)\n
This notebook is for offline training only. Backend runtime must only load trained artifacts.

In [1]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 200)

DATASET_ID = "bvk/CIC-MalMem-2022"
SAMPLE_SIZE: Optional[int] = 20_000
RANDOM_STATE = 42

FEATURE_PREFIXES = (
    "pslist.",
    "dlllist.",
    "handles.",
    "malfind.",
    "psxview.",
    "svcscan.",
    "callbacks.",
)

CACHE_DIR = Path("./.cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FULL_CACHE_FILE = CACHE_DIR / "cicmalmem_full.parquet"
SAMPLE_CACHE_FILE = CACHE_DIR / f"cicmalmem_sample_{SAMPLE_SIZE or 'full'}.parquet"


def _to_binary_class(value) -> int:
    text = str(value).strip().lower()
    return 0 if text == "benign" else 1


def _load_full_df() -> pd.DataFrame:
    if FULL_CACHE_FILE.exists():
        print(f"[cicmalmem] cache hit: {FULL_CACHE_FILE}")
        return pd.read_parquet(FULL_CACHE_FILE)

    print("[cicmalmem] downloading full train split from Hugging Face...")
    ds = load_dataset(DATASET_ID, split="train")
    df = ds.to_pandas()
    df.to_parquet(FULL_CACHE_FILE, index=False)
    print(f"[cicmalmem] cached full dataset to {FULL_CACHE_FILE}")
    return df


def _get_sample_df(full_df: pd.DataFrame, sample_size: Optional[int]) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(full_df):
        return full_df.copy()

    if SAMPLE_CACHE_FILE.exists():
        print(f"[cicmalmem] sample cache hit: {SAMPLE_CACHE_FILE}")
        return pd.read_parquet(SAMPLE_CACHE_FILE)

    labels = full_df["Class"].map(_to_binary_class)
    sampled, _ = train_test_split(
        full_df,
        train_size=sample_size,
        random_state=RANDOM_STATE,
        stratify=labels,
    )
    sampled = sampled.reset_index(drop=True)
    sampled.to_parquet(SAMPLE_CACHE_FILE, index=False)
    print(f"[cicmalmem] cached stratified sample to {SAMPLE_CACHE_FILE}")
    return sampled


full_df = _load_full_df()
cicmalmem_df_seed = _get_sample_df(full_df, SAMPLE_SIZE)

print("[cicmalmem] full rows:", len(full_df))
print("[cicmalmem] sampled rows:", len(cicmalmem_df_seed))
print("[cicmalmem] class distribution:\n", cicmalmem_df_seed["Class"].value_counts(dropna=False))

c:\Users\Zayad\Documents\FYP\01_WORK\ACDS-FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[cicmalmem] downloading full train split from Hugging Face...


c:\Users\Zayad\Documents\FYP\01_WORK\ACDS-FYP\.venv\Lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Zayad\.cache\huggingface\hub\datasets--bvk--CIC-MalMem-2022. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 58596/58596 [00:01<00:00, 50752.96 examples/s]


[cicmalmem] cached full dataset to .cache\cicmalmem_full.parquet
[cicmalmem] cached stratified sample to .cache\cicmalmem_sample_20000.parquet
[cicmalmem] full rows: 58596
[cicmalmem] sampled rows: 20000
[cicmalmem] class distribution:
 Class
Malware    10000
Benign     10000
Name: count, dtype: int64


In [2]:
# -----------------------------
# CIC-MalMem cleaning + feature prep
# -----------------------------

required_cols = {"Class", "Category", "Filename"}
missing_required = required_cols.difference(set(cicmalmem_df_seed.columns))
if missing_required:
    raise RuntimeError(f"Missing required columns: {sorted(missing_required)}")

cic_df = cicmalmem_df_seed.copy()
cic_df["label"] = cic_df["Class"].map(_to_binary_class)

behavioral_columns = [
    c for c in cic_df.columns if any(c.startswith(prefix) for prefix in FEATURE_PREFIXES)
]
if not behavioral_columns:
    raise RuntimeError("No CIC-MalMem behavioral feature columns were found")

X_raw = cic_df[behavioral_columns].copy()
for col in X_raw.columns:
    X_raw[col] = pd.to_numeric(X_raw[col], errors="coerce")

X_raw = X_raw.fillna(X_raw.median(numeric_only=True))

low_variance_cols = [
    c for c in X_raw.columns
    if X_raw[c].nunique(dropna=True) <= 1 or float(X_raw[c].var(ddof=0)) < 1e-5
]
X = X_raw.drop(columns=low_variance_cols).copy()
y = cic_df["label"].astype(int)

print("[cicmalmem] raw behavioral feature count:", len(behavioral_columns))
print("[cicmalmem] removed low-variance features:", len(low_variance_cols))
print("[cicmalmem] final feature count:", X.shape[1])
print("[cicmalmem] X shape:", X.shape, "| y shape:", y.shape)
print("[cicmalmem] label distribution:\n", y.value_counts(dropna=False))
print("[cicmalmem] remaining nulls in X:", int(X.isna().sum().sum()))

[cicmalmem] raw behavioral feature count: 48
[cicmalmem] removed low-variance features: 5
[cicmalmem] final feature count: 43
[cicmalmem] X shape: (20000, 43) | y shape: (20000,)
[cicmalmem] label distribution:
 label
1    10000
0    10000
Name: count, dtype: int64
[cicmalmem] remaining nulls in X: 0


## Train and evaluate a single behavioral malware model (CIC-MalMem)

Pipeline:
1. Load and sample CIC-MalMem behavioral telemetry features
2. Median-impute missing values and remove low-variance columns
3. Scale feature matrix with StandardScaler
4. Train one RandomForestClassifier (`class_weight='balanced'`)
5. Save model, scaler, and metadata for backend runtime

The runtime model is used only by the dedicated `/scan/behavioral` API endpoint.

In [3]:
from pathlib import Path
import json
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ARTIFACT_DIR = Path("../backend/ml/models")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifact directory:", ARTIFACT_DIR.resolve())

Artifact directory: C:\Users\Zayad\Documents\FYP\01_WORK\ACDS-FYP\backend\ml\models


In [4]:
# -----------------------------
# Train/test split + scaling
# -----------------------------
if len(X) == 0:
    raise RuntimeError("No training rows after preprocessing")

label_counts = y.value_counts(dropna=False)
can_stratify = len(label_counts) > 1 and label_counts.min() >= 2

split_kwargs = {
    "test_size": 0.2,
    "random_state": RANDOM_STATE,
}
if can_stratify:
    split_kwargs["stratify"] = y
else:
    print("[cicmalmem] class counts too small for stratified split; using non-stratified split")

X_train, X_test, y_train, y_test = train_test_split(X, y, **split_kwargs)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

scaler_path = ARTIFACT_DIR / "malware_scaler.pkl"
joblib.dump(scaler, scaler_path)
print("Saved scaler ->", scaler_path.resolve())

Train shape: (16000, 43)
Test shape: (4000, 43)
Saved scaler -> C:\Users\Zayad\Documents\FYP\01_WORK\ACDS-FYP\backend\ml\models\malware_scaler.pkl


In [5]:
# -----------------------------
# Train single CIC-MalMem behavioral malware model
# -----------------------------
behavioral_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

behavioral_model.fit(X_train_scaled, y_train)
y_pred = behavioral_model.predict(X_test_scaled)

behavioral_metrics = {
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1_score": float(f1_score(y_test, y_pred, zero_division=0)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
}

print("[cicmalmem] Metrics:", behavioral_metrics)
print("[cicmalmem] Classification report:\n", classification_report(y_test, y_pred, digits=4, zero_division=0))

model_path = ARTIFACT_DIR / "malware_model.pkl"
joblib.dump(behavioral_model, model_path)
print("Saved model ->", model_path.resolve())

[cicmalmem] Metrics: {'precision': 0.999000499750125, 'recall': 0.9995, 'f1_score': 0.9992501874531368, 'confusion_matrix': [[1998, 2], [1, 1999]]}
[cicmalmem] Classification report:
               precision    recall  f1-score   support

           0     0.9995    0.9990    0.9992      2000
           1     0.9990    0.9995    0.9993      2000

    accuracy                         0.9992      4000
   macro avg     0.9993    0.9992    0.9992      4000
weighted avg     0.9993    0.9992    0.9992      4000

Saved model -> C:\Users\Zayad\Documents\FYP\01_WORK\ACDS-FYP\backend\ml\models\malware_model.pkl


In [6]:
# -----------------------------
# Save single-model metadata
# -----------------------------
model_info = {
    "module": "malware_detection",
    "version": "3.0.0",
    "architecture": "cicmalmem_behavioral",
    "trained_at": pd.Timestamp.utcnow().isoformat(),
    "dataset": {
        "source": DATASET_ID,
        "sample_size": int(len(y)),
        "full_rows": int(len(full_df)),
    },
    "model": {
        "algorithm": "RandomForestClassifier",
        "model_path": str(model_path),
        "scaler_path": str(scaler_path),
        "feature_columns": list(X.columns),
        "metrics": behavioral_metrics,
    },
}

model_info_path = ARTIFACT_DIR / "malware_model_info.json"
with open(model_info_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

print("Saved metadata ->", model_info_path.resolve())
model_info

Saved metadata -> C:\Users\Zayad\Documents\FYP\01_WORK\ACDS-FYP\backend\ml\models\malware_model_info.json


{'module': 'malware_detection',
 'version': '3.0.0',
 'architecture': 'cicmalmem_behavioral',
 'trained_at': '2026-05-08T06:16:08.855304+00:00',
 'dataset': {'source': 'bvk/CIC-MalMem-2022',
  'sample_size': 20000,
  'full_rows': 58596},
 'model': {'algorithm': 'RandomForestClassifier',
  'model_path': '..\\backend\\ml\\models\\malware_model.pkl',
  'scaler_path': '..\\backend\\ml\\models\\malware_scaler.pkl',
  'feature_columns': ['pslist.nproc',
   'pslist.nppid',
   'pslist.avg_threads',
   'pslist.avg_handlers',
   'dlllist.ndlls',
   'dlllist.avg_dlls_per_proc',
   'handles.nhandles',
   'handles.avg_handles_per_proc',
   'handles.nfile',
   'handles.nevent',
   'handles.ndesktop',
   'handles.nkey',
   'handles.nthread',
   'handles.ndirectory',
   'handles.nsemaphore',
   'handles.ntimer',
   'handles.nsection',
   'handles.nmutant',
   'malfind.ninjections',
   'malfind.commitCharge',
   'malfind.protection',
   'malfind.uniqueInjections',
   'psxview.not_in_pslist',
   'psxvie